# Physics-Informed Deep Learning for Entropy Prediction in Heterogeneous Systems (caso termodinamico: CSTR)

**Paper:** Sahoo, B., Patra, D. (2026). *Physics-Informed Deep Learning for Entropy Prediction in Heterogeneous Systems: Thermodynamic and Information-Theoretic Case Studies.* arXiv:2606.01179 [cs.LG].

**Carpeta origen:** `PINNs/2. Termidinamica y cinetica/Physics-Informed Deep Learning for Entropy Prediction in Heterogeneous Systems Thermodynamic and Information-Theoretic Case Studies.pdf`

## Como se usan las PINNs en este paper

El paper presenta un framework **PIDL** (Physics-Informed Deep Learning) que fuerza simultaneamente restricciones fisicas de dominios heterogeneos (ODEs termodinamicas + PDE de Fokker-Planck) en una arquitectura compartida, usando la perdida general (Eq. 1):

$$\mathcal{L}_{total}=w_d\mathcal{L}_{data}+w_r\mathcal{L}_{res}+w_b\mathcal{L}_{BC}+w_i\mathcal{L}_{IC}$$

Este cuaderno reproduce el **caso de estudio termodinamico (Variante I)**: una PINN estima la concentracion $C_A(t)$, temperatura $T(t)$ y la tasa de generacion de entropia $\sigma(t)$ de un **reactor de tanque agitado continuo (CSTR)** con reaccion exotermica irreversible $A\to B$, resolviendo el sistema de EDOs acopladas (Eq. 4-6):

$$\frac{dC_A}{dt}=\frac{C_{A0}-C_A}{\tau}-k(T)C_A,\qquad \frac{dT}{dt}=\frac{T_0-T}{\tau}+\frac{-\Delta H_r}{\rho c_p}k(T)C_A-\frac{UA}{V\rho c_p}(T-T_c)$$
$$k(T)=k_0\exp\!\Big(-\frac{E_a}{RT}\Big)$$

con la tasa de generacion de entropia (Eq. 7):

$$\sigma(t)=-\frac{\Delta G_r(T)}{T}k(T)C_A+\frac{UA(T-T_c)^2}{VTT_c}$$

**El punto metodologico central**: la salida de entropia de la red pasa por una activacion **Softplus**, imponiendo $\sigma\geq0$ (Segunda Ley de la Termodinamica) **por construccion arquitectonica**, en vez de como penalizacion blanda que podria violarse. La red (6 capas x 128 neuronas, tanh, Eq. 14) predice $y=[C_A,T,\sigma]$, entrenada con la perdida multi-objetivo (Eq. 15-18):

$$\mathcal{L}_{total}^{thermo}=\lambda_d\mathcal{L}_{data}+\lambda_r\mathcal{L}_{ODE}+\lambda_\sigma\mathcal{L}_\sigma+\lambda_i\mathcal{L}_{IC}$$

donde $\mathcal{L}_\sigma$ fuerza consistencia entre la $\sigma$ de salida de la red (con restriccion dura $\geq0$) y el valor $\sigma$ calculado fisicamente (Eq. 7) a partir de las $(C_A,T)$ predichas por la misma red. El paper reporta que este framework **nunca viola la Segunda Ley** bajo ninguna condicion de prueba, y retiene >90% de precision usando solo 30% de los datos de entrenamiento.

Este cuaderno reproduce fielmente la arquitectura, la restriccion Softplus, la funcion de perdida multi-objetivo y el escenario de datos escasos (30% de entrenamiento) con los parametros exactos de la Tabla 1 del paper.

**Nota de interpretacion:** el paper define $\Delta G_r(T)=\Delta H_r-T\Delta S_r$ pero la Tabla 1 solo reporta $\Delta H_r$ (no $\Delta S_r$). Siguiendo la propia simplificacion implicita del paper (que solo exige $\Delta G_r<0$, sin usar $\Delta S_r$ en ningun otro lugar), aqui se aproxima $\Delta G_r(T)\approx\Delta H_r$ (constante) para el calculo de $\sigma$.

## Repositorio publico de referencia

El PDF no incluye un repositorio de codigo propio, ni se encontro uno especifico al buscar en GitHub. El paper se basa explicitamente en el framework PINN original de Raissi et al. (citado como referencia [10]):

- **maziarraissi/PINNs** &mdash; https://github.com/maziarraissi/PINNs

In [ ]:
# Instalacion de dependencias (ejecutar si no estan ya instaladas en el entorno)
%pip install -q torch numpy matplotlib

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Parametros del CSTR (Tabla 1) y solucion de referencia via RK4

In [ ]:
CA0 = 2.0        # concentracion de alimentacion [mol/m^3]
T0 = 300.0        # temperatura de alimentacion [K]
Tc = 280.0        # temperatura del refrigerante [K]
tau = 100.0       # tiempo de residencia [s]
k0 = 7.2e10       # factor pre-exponencial [1/s]
Ea = 7.27e4       # energia de activacion [J/mol]
dHr = -4.78e4     # entalpia de reaccion (exotermica, J/mol); -dHr = 4.78e4 en la Tabla 1
rho = 1000.0      # densidad [kg/m^3]
cp = 4.184e3      # capacidad calorifica [J/(kg K)]
UA_V = 1.678e3    # UA/V [W/(m^3 K)]
R_gas = 8.314
t_max = 500.0

def k_rate(T):
    return k0 * np.exp(-Ea / (R_gas * T))

def cstr_rhs(state):
    CA, T = state
    dCA = (CA0 - CA) / tau - k_rate(T) * CA
    dT = (T0 - T) / tau + (-dHr / (rho * cp)) * k_rate(T) * CA - (UA_V / (rho * cp)) * (T - Tc)
    return np.array([dCA, dT])

def rk4_solve(n_steps=5000, dt=0.1):
    state = np.array([CA0, T0])  # condicion inicial: reactor arranca en las condiciones de alimentacion
    traj = [state.copy()]
    for _ in range(n_steps):
        k1 = cstr_rhs(state)
        k2 = cstr_rhs(state + 0.5 * dt * k1)
        k3 = cstr_rhs(state + 0.5 * dt * k2)
        k4 = cstr_rhs(state + dt * k3)
        state = state + (dt / 6) * (k1 + 2 * k2 + 2 * k3 + k4)
        traj.append(state.copy())
    return np.array(traj)

n_steps = 2000
dt_sim = t_max / n_steps
traj = rk4_solve(n_steps=n_steps, dt=dt_sim)
t_ref = np.linspace(0, t_max, n_steps + 1)
CA_ref, T_ref = traj[:, 0], traj[:, 1]

def sigma_eq7(CA, T):
    """Eq. (7), con dG_r(T) aproximado por dH_r (ver nota de interpretacion arriba)."""
    return (-dHr / T) * k_rate(T) * CA + UA_V * (T - Tc)**2 / (T * Tc)

sigma_ref = sigma_eq7(CA_ref, T_ref)

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
axes[0].plot(t_ref, CA_ref); axes[0].set_title('$C_A(t)$ referencia (RK4)'); axes[0].set_xlabel('t [s]')
axes[1].plot(t_ref, T_ref); axes[1].set_title('$T(t)$ referencia (RK4)'); axes[1].set_xlabel('t [s]')
axes[2].plot(t_ref, sigma_ref); axes[2].set_title('$\\sigma(t)$ referencia (Eq. 7)'); axes[2].set_xlabel('t [s]')
plt.tight_layout(); plt.show()

## 2. Red PINN con restriccion dura Softplus en $\sigma$ (Eq. 14, Seccion 4.1)

In [ ]:
class ThermoPINN(nn.Module):
    def __init__(self, n_hidden=6, n_neurons=128):
        super().__init__()
        layers = [nn.Linear(1, n_neurons), nn.Tanh()]
        for _ in range(n_hidden - 1):
            layers += [nn.Linear(n_neurons, n_neurons), nn.Tanh()]
        layers += [nn.Linear(n_neurons, 3)]  # CA, T, sigma_raw
        self.net = nn.Sequential(*layers)
        self.softplus = nn.Softplus()

    def forward(self, t):
        out = self.net(t / t_max)
        CA = self.softplus(out[:, 0:1])            # CA >= 0 (fisicamente necesario)
        # T varia muy poco en torno a T0 (apenas unos pocos K, dada la cinetica de este caso);
        # se centra la salida en T0 con una escala estrecha para que la red pueda resolver
        # esa variacion fina sin necesitar pesos extremadamente pequenos.
        T = T0 + out[:, 1:2] * 20.0
        sigma = self.softplus(out[:, 2:3])          # restriccion dura: Segunda Ley, sigma >= 0
        return CA, T, sigma


model = ThermoPINN(n_hidden=4, n_neurons=64).to(device)  # version reducida para agilizar el entrenamiento


def d_dt(f, t):
    return torch.autograd.grad(f, t, grad_outputs=torch.ones_like(f),
                                create_graph=True, retain_graph=True)[0]

## 3. Perdida multi-objetivo con datos escasos (30% de la trayectoria, Eq. 15-18)

In [ ]:
frac_data = 0.30  # solo 30% de los datos disponibles, como en el escenario de bajo-dato del paper
n_data = int(frac_data * len(t_ref))
idx_data = np.sort(np.random.choice(len(t_ref), n_data, replace=False))

t_data = torch.tensor(t_ref[idx_data], dtype=torch.float32, device=device).view(-1, 1)
CA_data = torch.tensor(CA_ref[idx_data], dtype=torch.float32, device=device).view(-1, 1)
T_data = torch.tensor(T_ref[idx_data], dtype=torch.float32, device=device).view(-1, 1)

t_col = torch.linspace(1e-2, t_max, 2000, device=device).view(-1, 1).requires_grad_(True)
t_ic = torch.zeros(1, 1, device=device, requires_grad=True)


def k_rate_t(T):
    return k0 * torch.exp(-Ea / (R_gas * T))


def compute_loss(model, lam_d=50.0, lam_r=1.0, lam_sigma=1.0, lam_i=50.0):
    # L_data (Eq. 16)
    CA_p, T_p, _ = model(t_data)
    loss_data = torch.mean((CA_p - CA_data)**2) + torch.mean((T_p - T_data)**2)

    # L_ODE (Eq. 17)
    CA_c, T_c, sigma_c = model(t_col)
    dCA_dt = d_dt(CA_c, t_col)
    dT_dt = d_dt(T_c, t_col)
    k_c = k_rate_t(T_c)
    res_CA = dCA_dt - ((CA0 - CA_c) / tau - k_c * CA_c)
    res_T = dT_dt - ((T0 - T_c) / tau + (-dHr / (rho * cp)) * k_c * CA_c - (UA_V / (rho * cp)) * (T_c - Tc))
    loss_ode = torch.mean(res_CA**2) + torch.mean(res_T**2)

    # L_sigma (Eq. 18): consistencia entre sigma (softplus) y sigma fisica (Eq. 7)
    sigma_phys = (-dHr / T_c) * k_c * CA_c + UA_V * (T_c - Tc)**2 / (T_c * Tc)
    loss_sigma = torch.mean((sigma_c - sigma_phys)**2)

    # L_IC
    CA_0, T_0p, _ = model(t_ic)
    loss_ic = (CA_0 - CA0)**2 + (T_0p - T0)**2

    total = lam_d * loss_data + lam_r * loss_ode + lam_sigma * loss_sigma + lam_i * loss_ic.squeeze()
    return total, loss_data.item(), loss_ode.item(), loss_sigma.item()

## 4. Entrenamiento

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
history = []
for epoch in range(4000):
    optimizer.zero_grad()
    loss, l_d, l_r, l_s = compute_loss(model)
    loss.backward()
    optimizer.step()
    history.append(loss.item())
    if epoch % 500 == 0:
        print(f'epoch {epoch:5d} | loss={loss.item():.4e} | data={l_d:.4e} | ode={l_r:.4e} | sigma_consist={l_s:.4e}')

## 5. Resultados: reconstruccion con 30% de los datos y verificacion de la Segunda Ley

In [ ]:
t_plot = torch.linspace(0, t_max, 500, device=device).view(-1, 1)
with torch.no_grad():
    CA_pred, T_pred, sigma_pred = model(t_plot)

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
axes[0].plot(t_ref, CA_ref, label='RK4 (referencia)')
axes[0].plot(t_plot.cpu(), CA_pred.cpu(), '--', label='PINN (30% datos)')
axes[0].scatter(t_data.cpu(), CA_data.cpu(), s=5, color='k', alpha=0.3, label='datos de entrenamiento')
axes[0].set_title('$C_A(t)$'); axes[0].legend(fontsize=7)

axes[1].plot(t_ref, T_ref, label='RK4')
axes[1].plot(t_plot.cpu(), T_pred.cpu(), '--', label='PINN')
axes[1].set_title('$T(t)$'); axes[1].legend(fontsize=7)

axes[2].plot(t_ref, sigma_ref, label='Eq. (7), referencia')
axes[2].plot(t_plot.cpu(), sigma_pred.cpu(), '--', label='PINN (Softplus, >=0)')
axes[2].axhline(0, color='r', linestyle=':', linewidth=1)
axes[2].set_title('$\\sigma(t)$'); axes[2].legend(fontsize=7)
plt.tight_layout(); plt.show()

print(f'Minimo de sigma predicho: {sigma_pred.min().item():.6f} (debe ser >= 0 por construccion)')
r2_CA = 1 - np.sum((CA_pred.cpu().numpy().flatten() - np.interp(t_plot.cpu().numpy().flatten(), t_ref, CA_ref))**2) / \
    np.sum((np.interp(t_plot.cpu().numpy().flatten(), t_ref, CA_ref) - CA_ref.mean())**2)
print(f'R^2 aproximado de C_A con solo {frac_data*100:.0f}% de los datos: {r2_CA*100:.2f}%')